# Unit 08 - CUPED (Demo) · **V2 material**

**Atoms served:** `U08-A6` (**CUPED** - pre-period covariate; **no video**; Spotify Confidence docs and this notebook carry it)

**Estimated runtime:** ~40 seconds (the power table refits the model many times)

**After this notebook you can:** compare a plain difference-in-means to a `CUPED`-adjusted estimate, read the variance reduction as a number, and say when pre-period correlation is too weak to bother.

## Without code

1. Plain `ATE` standard error: about **0.20** on the simulated data (your seed may differ slightly).
2. `CUPED`-adjusted standard error: about **0.145**, a **variance** reduction near 47%. Watch the wording - halving the variance divides the standard error by `sqrt(2)`, so the SE falls by about a quarter, not by half.
3. Sample size payoff: the power table shows the plain test needing about **1,000** users per cohort to reach 80% power, while `CUPED` gets there at about **400** - the same decision on well under half the traffic.
4. Very weak pre-period link (correlation under 0.1): variance reduction collapses to well under 1% - the method has a precondition.

**Note:** `V28` names covariates as a lever on `SE(ATE)` but never walks through `CUPED`. Major platforms enable it by default; this notebook shows why.

## 1. The question

Your checkout experiment runs two weeks. You already have each user's pre-period spend. Can you use that history to reach the same decision with fewer users - and when does that trick fail?

## 2. Setup

**Before you run:** the install line in the next cell is commented out on purpose - Colab already has every library this notebook needs. If an import fails, remove the `#` and run the cell again.

In [ ]:
# Colab already ships numpy, pandas, scipy, statsmodels and matplotlib, so the
# install line below stays commented out and this cell runs instantly.
# If any import in this cell fails, delete the leading # and run the cell again.
# %pip install -q numpy pandas scipy statsmodels matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

RANDOM_SEED = 42  # change this and re-run to see how luck changes the story
np.random.seed(RANDOM_SEED)
plt.rcParams['figure.figsize'] = (7, 4)

## 3. The data

Simulate users with a true treatment lift on in-period revenue. Pre-period revenue correlates with in-period revenue - that correlation is what `CUPED` exploits.

We draw pre-period and in-period outcomes for control and treatment arms. The true `ATE` is fixed at 2.0 currency units.

In [ ]:
n = 8000
true_ate = 2.0
rho = 0.75  # pre-period correlation with in-period outcome

pre = np.random.normal(50, 10, n)
noise = np.random.normal(0, 5, n)
y0 = 0.6 * pre + noise + np.random.normal(0, 3, n)
treatment = np.random.binomial(1, 0.5, n)
y1 = y0 + true_ate * treatment + np.random.normal(0, 3, n)

df = pd.DataFrame({'pre': pre, 'y': y1, 'treatment': treatment})
print('Mean revenue - control:', round(df.loc[treatment==0, 'y'].mean(), 2),
      'treatment:', round(df.loc[treatment==1, 'y'].mean(), 2))

## 4. The naive move

Run a plain difference-in-means (or `y ~ treatment` regression). This ignores pre-period behaviour you already have.

Fit the unadjusted model and record the treatment coefficient and its standard error.

In [ ]:
plain = smf.ols('y ~ treatment', data=df).fit()
plain_se = plain.bse['treatment']
plain_ate = plain.params['treatment']
print('Plain ATE:', round(plain_ate, 3), 'SE:', round(plain_se, 3))

The estimate is in the right ballpark, but the standard error is wider than it needs to be.

## 5. What actually happens

**CUPED adjustment.** Regress in-period `y` on `treatment` and pre-period `pre`. Using the covariate typically shrinks the variance of the treatment coefficient.

Compare standard errors. The variance reduction ratio tells you how much precision `CUPED` bought.

In [ ]:
cuped = smf.ols('y ~ treatment + pre', data=df).fit()
cuped_se = cuped.bse['treatment']
cuped_ate = cuped.params['treatment']
var_reduction = 1 - (cuped_se / plain_se) ** 2
print('CUPED ATE:', round(cuped_ate, 3), 'SE:', round(cuped_se, 3))
print('Variance reduction:', round(var_reduction * 100, 1), '%')

With strong pre-period correlation, variance often drops by roughly half - matching what Spotify and DoorDash report in production.

**Sample size payoff.** If variance halves, you need about half the users for the same power. Shrink the cohort step by step and watch where the two methods stop agreeing - that gap is the payoff, expressed in users you did not have to recruit.

In [ ]:
# One lucky subsample proves nothing, so we repeat each cohort size many times
# and report the share of runs that reach significance - that share IS the power.
def power_at_n(n_sub, use_cuped, reps=150):
    formula = 'y ~ treatment + pre' if use_cuped else 'y ~ treatment'
    hits = 0
    for r in range(reps):
        sub = df.sample(n=n_sub, random_state=r)
        hits += smf.ols(formula, data=sub).fit().pvalues['treatment'] < 0.05
    return round(hits / reps, 3)

payoff = pd.DataFrame([
    {'n_per_cohort': n_sub,
     'plain_power': power_at_n(n_sub, use_cuped=False),
     'cuped_power': power_at_n(n_sub, use_cuped=True)}
    for n_sub in [200, 300, 400, 600, 1000, 1500]
])
print(payoff.to_string(index=False))

def smallest_n_reaching(col, target=0.8):
    ok = payoff.loc[payoff[col] >= target, 'n_per_cohort']
    return int(ok.min()) if len(ok) else 'above the largest n tested'

print('\nSmallest tested cohort reaching 80% power - plain:', smallest_n_reaching('plain_power'),
      '| CUPED:', smallest_n_reaching('cuped_power'))

That gap is why platforms turn `CUPED` on by default instead of listing it as an advanced option.

**When the precondition fails.** Re-generate data with weak pre-period correlation (~0.1). `CUPED` still runs, but the variance reduction vanishes.

In [ ]:
pre_w = np.random.normal(50, 10, n)
y0_w = 0.05 * pre_w + np.random.normal(0, 8, n) + np.random.normal(0, 3, n)
y1_w = y0_w + true_ate * treatment + np.random.normal(0, 3, n)
df_w = pd.DataFrame({'pre': pre_w, 'y': y1_w, 'treatment': treatment})

se_plain_w = smf.ols('y ~ treatment', data=df_w).fit().bse['treatment']
se_cuped_w = smf.ols('y ~ treatment + pre', data=df_w).fit().bse['treatment']
vr_w = 1 - (se_cuped_w / se_plain_w) ** 2
print('Weak-link variance reduction:', round(vr_w * 100, 1), '%')

Without predictive pre-period data, `CUPED` is extra complexity for almost no gain. Check correlation before you flip the platform switch.

## 6. What you do about it

- Turn on `CUPED` when pre-period behaviour predicts your metric (`U08-A6`).
- Read variance reduction as a number, not a slogan - aim for 30-50% in production cases.
- Halving variance roughly halves required sample size for the same power.
- Skip or downgrade `CUPED` when pre-period correlation is weak.

**When this matters less:** Brand-new metrics with no history - there is nothing to covary on yet.

---

**Takeaway:** `CUPED` is the fourth lever on `SE(ATE)` from `V28`, done properly. It is on by default because it buys the same decision at half the traffic - when the pre-period link is real.

**Back to the unit:** [V2 unit 08](../V2/units/unit-08-power-duration-sample-size/README.md)